In [1]:
import os
import sys
import pandas as pd
import numpy as np

# Add project root so we can import src
ROOT_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
if ROOT_DIR not in sys.path:
    sys.path.append(ROOT_DIR)

from src.data_enrichment import get_features

# Modelos
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# Preprocesado y métricas
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import (
    roc_auc_score,
    recall_score,
    precision_score,
    f1_score,
    confusion_matrix,
    classification_report,
)

# SMOTE + pipeline “correcta”
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

RAW_DIR = "../data/raw"

df_feats, _ = get_features(RAW_DIR)

print("Dataset shape:", df_feats.shape)

MIN_MINUTES = 100

# MISMO FILTRO que antes (2008–2024, mínimo de minutos)
df_ml = df_feats[
    (df_feats["season_end_year"] >= 2008)
    & (df_feats["season_end_year"] <= 2024)
    & (df_feats["minutes_played"] >= MIN_MINUTES)
].copy()

print("ML dataset:", df_ml.shape)
df_ml.head()


Dataset shape: (76833, 79)
ML dataset: (23442, 79)


,player_id,minutes_played,goals,assists,yellow_cards,second_yellow_cards,direct_red_cards,penalty_goals,matches_played,clean_sheets,...,won_champions,team_ucl_strength,Titles,win_rate,goals_per_game,num_trophies,ballon_dor_winner,player_name,position,main_position
11,10,201.0,10.0,8,2,0,0,0,27,0,...,0.0,0.000000,0.0,0.0,0.00,0.0,0,Miroslav Klose (10),Attack - Centre-Forward,Attack
12,10,283.0,17.0,10,3,0,0,2,34,0,...,0.0,0.000000,0.0,0.0,0.00,0.0,0,Miroslav Klose (10),Attack - Centre-Forward,Attack
13,10,679.0,4.0,1,3,0,0,0,33,0,...,0.0,0.000000,0.0,0.0,0.00,0.0,0,Miroslav Klose (10),Attack - Centre-Forward,Attack
14,10,972.0,2.0,1,2,0,0,0,22,0,...,0.0,0.000000,0.0,0.0,0.00,0.0,0,Miroslav Klose (10),Attack - Centre-Forward,Attack
15,10,190.0,12.0,7,1,0,0,0,27,0,...,0.0,0.014706,0.0,0.4,1.55,0.0,0,Miroslav Klose (10),Attack - Centre-Forward,Attack


In [ ]:
# ===  FINAL FEATURES DE DIMENSIONALITY REDUCTION  ===
final_features = [
    # === Selected 30 ===
    'a_per90_z_lag1', 'ga_per90_z_lag1', 'matches_played_z_lag1',
    'g_per90_z_lag1', 'pen_share_z_lag1', 'g_per90_w', 'ga_per90_w',
    'a_per90_w', 'pen_share_w', 'a_per90_z_delta', 'ga_per90_z_delta',
    'main_position', 'g_per90_z_delta', 'age', 'win_rate', 'height',
    'goals_per_game', 'minutes_played_z_lag1', 'pen_share_z_delta',
    'age_norm', 'team_ucl_strength', 'age_penalty',
    'matches_played_z_delta', 'gc_per90_z_lag1',
    'minutes_played_z_delta', 'season_end_year',
    'Titles', 'num_trophies', 'won_champions',

    # === Metadata (not scaled) ===
    'player_id', 'player_name'
]

# Nos aseguramos de no tener duplicados
final_features = list(dict.fromkeys(final_features))

# Metadatos que NO deben entrar al modelo, solo para reportar
metadata_cols = [
    'player_id',
    'player_name',
    'season_end_year',
    'minutes_played',        # esta no venía en final_features, pero la queremos conservar
]

# Intersección con las columnas del df (por seguridad)
missing = [c for c in final_features if c not in df_ml.columns]
if missing:
    print("⚠️ WARNING - these final_features are missing in df_ml:", missing)

available = [c for c in final_features if c in df_ml.columns]

# Columnas que el modelo realmente va a usar
model_feature_cols = [
    c for c in available
    if c not in metadata_cols  # sacamos metadatos
]

print("Num model features:", len(model_feature_cols))
print(model_feature_cols)


Num model features: 28
['a_per90_z_lag1', 'ga_per90_z_lag1', 'matches_played_z_lag1', 'g_per90_z_lag1', 'pen_share_z_lag1', 'g_per90_w', 'ga_per90_w', 'a_per90_w', 'pen_share_w', 'a_per90_z_delta', 'ga_per90_z_delta', 'main_position', 'g_per90_z_delta', 'age', 'win_rate', 'height', 'goals_per_game', 'minutes_played_z_lag1', 'pen_share_z_delta', 'age_norm', 'team_ucl_strength', 'age_penalty', 'matches_played_z_delta', 'gc_per90_z_lag1', 'minutes_played_z_delta', 'Titles', 'num_trophies', 'won_champions']


In [3]:
# Temporal split: HARD STRICT NON-LEAKING
df_train = df_ml[df_ml["season_end_year"] <= 2018].copy()
df_val   = df_ml[(df_ml["season_end_year"] >= 2019) & (df_ml["season_end_year"] <= 2022)].copy()
df_test  = df_ml[df_ml["season_end_year"] >= 2023].copy()

print("Train:", df_train.shape)
print("Val:", df_val.shape)
print("Test:", df_test.shape)

# Matrices de entrada/salida
X_train = df_train[model_feature_cols].copy()
y_train = df_train["ballon_dor_winner"].astype(int)

X_val = df_val[model_feature_cols].copy()
y_val = df_val["ballon_dor_winner"].astype(int)

X_test = df_test[model_feature_cols].copy()
y_test = df_test["ballon_dor_winner"].astype(int)

print("y_train balance:\n", y_train.value_counts())


Train: (14301, 79)
Val: (6038, 79)
Test: (3103, 79)
y_train balance:
 ballon_dor_winner
0    14290
1       11
Name: count, dtype: int64


In [4]:
# Detectar tipos de columnas
numeric_features = [
    c for c in model_feature_cols
    if df_ml[c].dtype != "object"
]

categorical_features = [
    c for c in model_feature_cols
    if df_ml[c].dtype == "object"
]

print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ],
    remainder="drop",
)

# Definimos los modelos base
models = {
    "lr": LogisticRegression(max_iter=2000, n_jobs=-1),
    "rf": RandomForestClassifier(
        n_estimators=500,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1,
    ),
    "xgb": XGBClassifier(
        n_estimators=600,
        learning_rate=0.05,
        max_depth=5,
        subsample=0.9,
        colsample_bytree=0.9,
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1,
    ),
}

# Creamos pipelines con preprocesado + SMOTE + modelo
pipelines = {}
val_results = {}

for name, clf in models.items():
    pipe = ImbPipeline(steps=[
        ("preprocess", preprocessor),
        ("smote", SMOTE(k_neighbors=1, random_state=42)),
        ("clf", clf),
    ])
    
    pipe.fit(X_train, y_train)  # aquí se ajusta scaler + onehot + SMOTE solo con TRAIN

    proba_val = pipe.predict_proba(X_val)[:, 1]
    pred_val = pipe.predict(X_val)

    val_auc = roc_auc_score(y_val, proba_val)
    val_rec = recall_score(y_val, pred_val, zero_division=0)
    val_prec = precision_score(y_val, pred_val, zero_division=0)
    val_f1 = f1_score(y_val, pred_val, zero_division=0)

    val_results[name] = {
        "AUC": val_auc,
        "Recall": val_rec,
        "Precision": val_prec,
        "F1": val_f1,
    }

    pipelines[name] = pipe

    print(f"\n=== {name.upper()} VALIDATION METRICS ===")
    print("AUC      :", val_auc)
    print("Recall   :", val_rec)
    print("Precision:", val_prec)
    print("F1       :", val_f1)

val_results


Numeric features: ['a_per90_z_lag1', 'ga_per90_z_lag1', 'matches_played_z_lag1', 'g_per90_z_lag1', 'pen_share_z_lag1', 'g_per90_w', 'ga_per90_w', 'a_per90_w', 'pen_share_w', 'a_per90_z_delta', 'ga_per90_z_delta', 'g_per90_z_delta', 'age', 'win_rate', 'height', 'goals_per_game', 'minutes_played_z_lag1', 'pen_share_z_delta', 'age_norm', 'team_ucl_strength', 'age_penalty', 'matches_played_z_delta', 'gc_per90_z_lag1', 'minutes_played_z_delta', 'Titles', 'num_trophies', 'won_champions']
Categorical features: ['main_position']

=== LR VALIDATION METRICS ===
AUC      : 0.9997790665562
Recall   : 1.0
Precision: 0.3
F1       : 0.46153846153846156

=== RF VALIDATION METRICS ===
AUC      : 0.9990610328638498
Recall   : 0.0
Precision: 0.0
F1       : 0.0

=== XGB VALIDATION METRICS ===
AUC      : 0.9996685998342999
Recall   : 0.3333333333333333
Precision: 0.3333333333333333
F1       : 0.3333333333333333


{'lr': {'AUC': 0.9997790665562,
  'Recall': 1.0,
  'Precision': 0.3,
  'F1': 0.46153846153846156},
 'rf': {'AUC': 0.9990610328638498, 'Recall': 0.0, 'Precision': 0.0, 'F1': 0.0},
 'xgb': {'AUC': 0.9996685998342999,
  'Recall': 0.3333333333333333,
  'Precision': 0.3333333333333333,
  'F1': 0.3333333333333333}}

In [5]:
# Elegir mejor modelo por AUC
best_model_name = max(val_results, key=lambda k: val_results[k]["AUC"])
best_pipe = pipelines[best_model_name]

print("\n===================================")
print("BEST MODEL ON VALIDATION:", best_model_name.upper())
print(val_results[best_model_name])
print("===================================\n")

# Evaluación en TEST
proba_test = best_pipe.predict_proba(X_test)[:, 1]
pred_test = best_pipe.predict(X_test)

test_auc = roc_auc_score(y_test, proba_test)
test_rec = recall_score(y_test, pred_test, zero_division=0)
test_prec = precision_score(y_test, pred_test, zero_division=0)
test_f1 = f1_score(y_test, pred_test, zero_division=0)

print("=== TEST RESULTS (BEST MODEL) ===")
print("AUC      :", test_auc)
print("Recall   :", test_rec)
print("Precision:", test_prec)
print("F1       :", test_f1)
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, pred_test))



BEST MODEL ON VALIDATION: LR
{'AUC': 0.9997790665562, 'Recall': 1.0, 'Precision': 0.3, 'F1': 0.46153846153846156}

=== TEST RESULTS (BEST MODEL) ===
AUC      : 0.9492099322799098
Recall   : 0.0
Precision: 0.0
F1       : 0.0

Confusion Matrix:
[[3099    2]
 [   2    0]]


In [6]:
# Columnas de contexto que queremos para el ranking
output_meta_cols = [
    c for c in ["player_id", "player_name", "season_end_year", "minutes_played"]
    if c in df_test.columns
]

df_test_out = df_test[output_meta_cols].copy()
df_test_out["proba_win"] = proba_test

top_candidates = (
    df_test_out
    .sort_values("proba_win", ascending=False)
    .head(20)
)

top_candidates


,player_id,player_name,season_end_year,minutes_played,proba_win
44315,418560,Erling Haaland (418560),2023.0,148.0,0.995954
6157,132098,Harry Kane (132098),2024.0,212.0,0.995942
40166,38253,Robert Lewandowski (38253),2023.0,212.0,0.296656
26184,28003,Lionel Messi (28003),2023.0,332.0,0.089673
75010,93720,Alexandre Lacazette (93720),2023.0,109.0,0.025413
2083,111266,Anthony Losilla (111266),2024.0,2587.0,0.021099
34914,343537,Artem Dovbyk (343537),2024.0,109.0,0.016521
28801,29692,Lukasz Fabianski (29692),2023.0,3113.0,0.013912
34714,342229,Kylian Mbappé (342229),2023.0,190.0,0.010131
21935,23951,Steve Mandanda (23951),2024.0,3060.0,0.009719
